In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.config_loader as config_loader
import util.read_price as read_price
import src.ip.optimizations as optimizations

print("config_loader:", config_loader.__file__)
print("read_price:", read_price.__file__)
print("rolling_ce:", optimizations.__file__)


battery, markets = config_loader.load_config(PROJECT_ROOT / "configs" / "battery_config.yaml")
ip_cfg = markets["ip"]

ip_det_xgb  = read_price.read_ip_forecast(model="XGB",  kind="deterministic", freq="15min")
ip_det_lear = read_price.read_ip_forecast(model="LEAR", kind="deterministic", freq="15min")
ip_det_qr   = read_price.read_ip_forecast(model="QR",   kind="deterministic", freq="15min")

ip_real = read_price.read_ip_real_prices(freq="15min", keep_extra_columns=False)["Price"]

start_date = "2023-01-01"
end_date = "2023-12-01"




config_loader: c:\Users\mmascare\OneDrive - KU Leuven\Documents\Code\DA_Optimization\src\config_loader.py
read_price: c:\Users\mmascare\OneDrive - KU Leuven\Documents\Code\DA_Optimization\util\read_price.py
rolling_ce: c:\Users\mmascare\OneDrive - KU Leuven\Documents\Code\DA_Optimization\src\ip\optimizations.py
C:\Users\mmascare\OneDrive - KU Leuven\Documents\Code\DA_Optimization\Data\IP_CET\IP_XGB.csv
C:\Users\mmascare\OneDrive - KU Leuven\Documents\Code\DA_Optimization\Data\IP_CET\IP_LEAR.csv
C:\Users\mmascare\OneDrive - KU Leuven\Documents\Code\DA_Optimization\Data\IP_CET\IP_QR.csv
C:\Users\mmascare\OneDrive - KU Leuven\Documents\Code\DA_Optimization\Data\IP_CET\IP_Real_Prices.csv


In [2]:
terminal_penalty=0.1

res = optimizations.run_ip_rolling_ce_models(
    battery=battery,
    market=ip_cfg,
    forecasts={"lear": ip_det_lear, "qr": ip_det_qr}, #"xgb": ip_det_xgb, 
    real_price_series=ip_real,
    price_source=["forecast", "naive", "perfect_foresight"],
    start=start_date,
    end=end_date,
    terminal_target_kwh=battery.energy_kwh*0.5,
    terminal_penalty=terminal_penalty,
    terminal_penalty_mode="L1",
    solver_name="gurobi_direct",
    save=True,
    tag="v1",
)
df = res.history



Solving perfect_foresight: 100.0 %